In [6]:
import numpy as np

In [1]:
class LinearRegression:
    """
    线性回归，支持正规方程法和梯度下降法。
    
    模型假设: $y = x' w + b$，其中$w$为权重向量，$b$为截距
    """
    def __init__(self, fit_intercept=True, method='NormalEquation', learning_rate=0.01, max_iteration=1000, tol=1e-6):
        """
        参数:
        - fit_intercept:是否拟合截距$b$
        - method:'NormalEquation'(正规方程法)或'GradientDescent'(梯度下降法)
        - learning_rate:梯度下降学习率
        - max_iteration:梯度下降最大迭代次数
        - tol:梯度下降收敛容限(当损失变化小于tol时停止)
        """
        self.fit_intercept = fit_intercept
        self.method = method
        self.learning_rate = learning_rate
        self.max_iteration = max_iteration
        self.tol = tol
        
        self.coeffcient_ = None      # 权重向量$w$
        self.intercept_ = None       # 截距$b$ (仅当 fit_intercept=True)
        self.loss_history_ = []

In [8]:
def _add_intercept(self, X):
    """
    在$n \times d$矩阵
    \[
      X = \begin{pmatrix} x^{(1)} \\ x^{(2)} \\ \vdots \\ x^{(n)} \end{pmatrix} 
    \]
    的左侧拼接一个长度为$n$的元素全为$1$的列向量
    """
    return np.column_stack([np.ones(X.shape[0]), X]) # column_stack()将具有相同行数的矩阵水平拼接，X.shape[0]返回矩阵X的行数

In [9]:
def fit(self, X, y):
        """
        训练模型。
        X: 行数为样本数，列数为特征数(向量$x^{(i)}$的长度)的矩阵
        y: 长度为样本数的向量
        """
        if self.fit_intercept:
            X = self._add_intercept(X) 
        
        if self.method == 'NormalEquation':
            self._fit_NE(X, y)
        elif self.method == 'GradientDescent':
            self._fit_GD(X, y)
        else:
            raise ValueError("method 参数须为 'NormalEquation' 或 'GradientDescent'")
        
        if self.fit_intercept: # 分离出截距和权重
            self.intercept_ = self.coeffcient_[0]
            self.coefficient_ = self.coefficient_[1:]
        else:
            self.intercept_ = 0.0
        
        return self

In [10]:
def _fit_NE(self, X, y):
    """
    正规方程法：$\theta = (X' X)^{-1} X' y$
    """
    # 通过伪逆保证数值稳定性，等价于最小二乘解
    self.coefficient_ = np.linalg.pinv(X.T @ X) @ X.T @ y

In [11]:
def _fit_GD(self, X, y):
    """
    梯度下降法：$\theta \gets \theta - \alpha X' ( y - X \theta )$
    """
    n , d = X.shape
    self.coefficient_ = np.zeros(d)  
    prev_loss = float('inf')
        
    for i in range(self.max_iteration):
        # 初始值
        y_pred = X @ self.coeffcient_ 
        # 梯度向量
        grad = (X.T @ (y_pred - y))
        # $\theta \gets \theta - \alpha \nabla J ( \theta )$
        self.coefficient_ = self.coefficient_ - self.learning_rate * grad
            
        # 计算当前损失并记录
        loss = (1 / 2) * np.sum((y_pred - y) ** 2)
        self.loss_history_.append(loss)
            
        # 收敛判断
        if abs(prev_loss - loss) < self.tol: 
            print(f"梯度下降在第{i+1}轮收敛，损失: {loss:.6f}")
            break
        prev_loss = loss

In [12]:
def predict(self, X):
    """
    预测新数据。
    """
    if self.fit_intercept:
        X = self._add_intercept(X)
    return X @ np.r_[self.intercept_, self.coefficient_] if self.fit_intercept else X @ self.coeffcient_ # np.r_[] 将两个列数相同的矩阵进行垂直拼接